# Unidad 10 · Regresión lineal múltiple y diagnóstico

Cuaderno de la base abierta de ingeniería (Análisis de Datos). Reproduce el ejemplo resuelto (tiempo de
despacho según bultos y kilómetros) y la pieza del caso Estampa Sur (unidades vendidas según precio y
publicidad). `../datos/u10_ejemplo.csv` lo genera el verificador; `../datos/ventas.csv`, el script del caso.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

## 1. Ejemplo · mínimos cuadrados con matrices

$\hat\beta = (X^\top X)^{-1} X^\top y$, con la columna de unos para la constante.

In [ ]:
ej = pd.read_csv("../datos/u10_ejemplo.csv")
X = np.column_stack([np.ones(len(ej)), ej.bultos, ej.km])
y = ej.minutos.to_numpy(float)
XtX, Xty = X.T @ X, X.T @ y
inv = np.linalg.inv(XtX)
beta_ejemplo = inv @ Xty
e = y - X @ beta_ejemplo
n, k = X.shape
mce = e @ e / (n - k)
se = np.sqrt(mce * np.diag(inv))
print("X'X =\n", XtX, "\nX'y =", Xty)
print("beta =", beta_ejemplo.round(4), " ee =", se.round(4), " t =", (beta_ejemplo / se).round(2))

In [ ]:
mod_ej = smf.ols("minutos ~ bultos + km", data=ej).fit()
print(mod_ej.summary())
print(mod_ej.get_prediction(pd.DataFrame({"bultos": [5], "km": [8]})).summary_frame(alpha=0.05))

## 2. Caso Estampa Sur · unidades vendidas según precio y publicidad

In [ ]:
v = pd.read_csv("../datos/ventas.csv")
modelo_caso = smf.ols("unidades ~ precio + publicidad", data=v).fit()
print(modelo_caso.summary())
prediccion_caso = modelo_caso.get_prediction(pd.DataFrame({"precio": [11500], "publicidad": [100000]})).summary_frame(alpha=0.05)
print(prediccion_caso)

## 3. Diagnóstico

Normalidad (Shapiro), homocedasticidad (Breusch-Pagan), independencia en el tiempo (Durbin-Watson),
multicolinealidad (VIF) y puntos influyentes (palanca, residuo estudentizado y distancia de Cook).

In [ ]:
r = modelo_caso.resid
print("Shapiro:", stats.shapiro(r))
print("Breusch-Pagan (LM, p, F, p):", het_breuschpagan(r, modelo_caso.model.exog))
print("Durbin-Watson:", durbin_watson(r))
vif_caso = variance_inflation_factor(modelo_caso.model.exog, 1)
print("VIF precio = VIF publicidad =", vif_caso)
inf = modelo_caso.get_influence()
tabla_inf = pd.DataFrame({"semana": v.semana, "palanca": inf.hat_matrix_diag,
                          "r_estud": inf.resid_studentized_external, "cook": inf.cooks_distance[0]})
print(tabla_inf.sort_values("cook", ascending=False).head())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
ax[0].scatter(modelo_caso.fittedvalues, r); ax[0].axhline(0, color="k", lw=0.8); ax[0].set_xlabel("ajustado")
sm.qqplot(r, line="s", ax=ax[1])
ax[2].plot(v.semana, r, marker="o"); ax[2].axhline(0, color="k", lw=0.8); ax[2].set_xlabel("semana")
plt.tight_layout(); plt.show()

## 4. La regresión como modelo lineal general

Un GLM con familia normal y enlace identidad da los mismos coeficientes; su devianza es la suma de
cuadrados del error. Es el puente a la unidad 11.

In [ ]:
glm = smf.glm("unidades ~ precio + publicidad", data=v, family=sm.families.Gaussian()).fit()
print(glm.params, "\ndevianza =", glm.deviance, " SCE =", modelo_caso.ssr)